In [0]:
# ================================================================================
# PIPELINE DE VENDAS - ARQUITETURA MEDALLION (Bronze → Silver → Gold)
# ================================================================================
# Objetivo: Processar arquivos CSV de vendas mensais por vendedor
# Padrão de arquivo: SELLER_ID_YYYY_MM_sales.csv
# Exemplo: 1204_2024_01_sales.csv
#
# Camadas:
# - Bronze: Dados originais + metadados extraídos do nome do arquivo
# - Silver: Dados tratados, deduplica dos e validados
# - Gold: Star Schema + métricas para consumo analítico
# ================================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable
import re

# ========================================
# CONFIGURAÇÃO DO AMBIENTE
# ========================================

CATALOG = "workspace"
SCHEMA = "default"

# Caminhos dos volumes (em ordem de prioridade)
VOLUME_PATHS = [
    "/Volumes/workspace/default/dados_brutos/Dataset_problema_tecnico_rpe/",
    "/Volumes/workspace/default/dados_brutos/",
    "/Volumes/main/default/dados_brutos/"
]

# ========================================
# NOMES DAS TABELAS
# ========================================

# Camada Bronze
TBL_BRONZE_VENDAS = f"{CATALOG}.{SCHEMA}.bronze_vendas"

# Camada Silver
TBL_SILVER_VENDAS = f"{CATALOG}.{SCHEMA}.silver_vendas"
TBL_QUARENTENA = f"{CATALOG}.{SCHEMA}.quarentena_registros_invalidos"

# Camada Gold - Star Schema
TBL_DIM_PRODUTO = f"{CATALOG}.{SCHEMA}.dim_produto"
TBL_DIM_VENDEDOR = f"{CATALOG}.{SCHEMA}.dim_vendedor"
TBL_DIM_TEMPO = f"{CATALOG}.{SCHEMA}.dim_tempo"
TBL_FATO_VENDAS = f"{CATALOG}.{SCHEMA}.fato_vendas"

# Camada Gold - Métricas Agregadas
TBL_METRICAS_VENDAS = f"{CATALOG}.{SCHEMA}.gold_metricas_vendas"

# Tabela de Controle para Processamento Incremental
TBL_CONTROLE_ARQUIVOS = f"{CATALOG}.{SCHEMA}.controle_arquivos_processados"

print("="*80)
print(" ✅ CONFIGURAÇÃO CARREGADA COM SUCESSO")
print("="*80)
print(f"\n📂 Catálogo: {CATALOG}")
print(f"📂 Schema: {SCHEMA}")
print(f"\n📊 Tabelas do Pipeline:")
print(f"   🔷 Bronze: {TBL_BRONZE_VENDAS}")
print(f"   🔷 Silver: {TBL_SILVER_VENDAS}")
print(f"   🔷 Gold: {TBL_FATO_VENDAS}, {TBL_DIM_PRODUTO}, {TBL_DIM_VENDEDOR}, {TBL_DIM_TEMPO}")
print(f"\n🎯 Pronto para iniciar o pipeline!\n")

In [0]:
# ================================================================================
# CAMADA BRONZE - DADOS BRUTOS + METADADOS
# ================================================================================
# Objetivo:
# 1. Ingerir arquivos CSV sem transformações
# 2. Extrair metadados do nome do arquivo (seller_id, ano, mês)
# 3. Garantir idempotência (reprocessamento não duplica dados)
# ================================================================================

print("="*80)
print(" 🔷 CAMADA BRONZE - INGESTÃO DE DADOS BRUTOS")
print("="*80)

# ========================================
# FUNÇÕES AUXILIARES
# ========================================

def encontrar_caminho_vendas(caminhos):
    """Busca o primeiro caminho válido que contém arquivos *_sales.csv"""
    for path in caminhos:
        try:
            caminho_pattern = f"{path.rstrip('/')}/*_sales.csv"
            test_df = spark.read.format("csv").option("header", "true").load(caminho_pattern).limit(1)
            if test_df.count() > 0:
                return caminho_pattern, path
        except Exception:
            continue
    return None, None

def extrair_metadados_arquivo(file_path):
    """
    Extrai seller_id, ano e mês do nome do arquivo.
    
    Formatos aceitos:
    - seller_123_2025_01_sales.csv → seller_id=123, ano=2025, mes=01
    - 1204_2024_01_sales.csv → seller_id=1204, ano=2024, mes=01
    
    Retorna:
    - Dict com: seller_id_arquivo (string), ano_arquivo (int), mes_arquivo (int)
    """
    # Tenta formato com prefixo "seller_"
    match = re.search(r'seller_(\d+)_(\d{4})_(\d{2})_sales', file_path)
    if match:
        return {
            'seller_id_arquivo': match.group(1),
            'ano_arquivo': int(match.group(2)),
            'mes_arquivo': int(match.group(3))
        }
    
    # Tenta formato sem prefixo (apenas número)
    match = re.search(r'/(\d+)_(\d{4})_(\d{2})_sales', file_path)
    if match:
        return {
            'seller_id_arquivo': match.group(1),
            'ano_arquivo': int(match.group(2)),
            'mes_arquivo': int(match.group(3))
        }
    
    return {'seller_id_arquivo': None, 'ano_arquivo': None, 'mes_arquivo': None}

# ========================================
# 1. CRIAÇÃO DA TABELA DE CONTROLE (se não existir)
# ========================================

if not spark.catalog.tableExists(TBL_CONTROLE_ARQUIVOS):
    print("\n🆕 Criando tabela de controle de arquivos...")
    schema_controle = StructType([
        StructField("arquivo", StringType(), False),
        StructField("data_processamento", StringType(), False),
        StructField("qtd_registros", IntegerType(), True),
        StructField("status", StringType(), True)
    ])
    df_controle_vazio = spark.createDataFrame([], schema_controle)
    df_controle_vazio.write.format("delta").mode("overwrite").saveAsTable(TBL_CONTROLE_ARQUIVOS)
    print(f"✅ Tabela {TBL_CONTROLE_ARQUIVOS} criada")

# ========================================
# 2. LOCALIZAÇÃO E IDENTIFICAÇÃO DE ARQUIVOS
# ========================================

print("\n📂 Buscando arquivos de vendas...")
caminho_vendas, caminho_base = encontrar_caminho_vendas(VOLUME_PATHS)

if not caminho_vendas:
    raise FileNotFoundError("❌ Nenhum arquivo *_sales.csv encontrado nos volumes configurados")

print(f"✅ Arquivos localizados: {caminho_vendas}")

# Lista TODOS os arquivos disponíveis
df_todos_arquivos = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(caminho_vendas)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .select("_source_file")
    .distinct()
)

total_arquivos_disponiveis = df_todos_arquivos.count()
print(f"📁 Total de arquivos disponíveis: {total_arquivos_disponiveis}")

# ========================================
# 3. FILTRO INCREMENTAL - APENAS ARQUIVOS NOVOS
# ========================================

print("\n🔍 Identificando arquivos novos (não processados)...")

# Carrega lista de arquivos já processados
df_processados = spark.table(TBL_CONTROLE_ARQUIVOS).select("arquivo")

# Identifica arquivos NOVOS (ainda não processados)
df_arquivos_novos = df_todos_arquivos.join(
    df_processados,
    df_todos_arquivos._source_file == df_processados.arquivo,
    "left_anti"
)

qtd_arquivos_novos = df_arquivos_novos.count()

if qtd_arquivos_novos == 0:
    print("✅ Nenhum arquivo novo para processar. Pipeline já está atualizado!")
    print(f"\n📋 Resumo:")
    print(f"   • Arquivos disponíveis: {total_arquivos_disponiveis}")
    print(f"   • Arquivos já processados: {df_processados.count()}")
    print(f"   • Arquivos novos: 0")
    dbutils.notebook.exit("Nenhum arquivo novo para processar")

print(f"🆕 {qtd_arquivos_novos} arquivo(s) novo(s) identificado(s)")

# Lista os arquivos novos
arquivos_novos_list = [row._source_file for row in df_arquivos_novos.collect()]
for arquivo in arquivos_novos_list:
    print(f"   • {arquivo.split('/')[-1]}")

# ========================================
# 4. LEITURA INCREMENTAL - APENAS ARQUIVOS NOVOS
# ========================================

print("\n📂 Carregando APENAS arquivos novos...")

# Lê SOMENTE os arquivos novos
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(caminho_vendas)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .filter(F.col("_source_file").isin(arquivos_novos_list))
)

total_registros = df_raw.count()
print(f"📊 Total de registros nos arquivos novos: {total_registros:,}")

# ========================================
# 5. EXTRAÇÃO DE METADADOS DO NOME DO ARQUIVO
# ========================================

print("\n🔍 Extraindo metadados dos nomes dos arquivos (seller_id, ano, mês)...")

metadata_schema = StructType([
    StructField("seller_id_arquivo", StringType(), True),
    StructField("ano_arquivo", IntegerType(), True),
    StructField("mes_arquivo", IntegerType(), True)
])

extrair_metadados_udf = F.udf(extrair_metadados_arquivo, metadata_schema)

df_com_metadados = (
    df_raw
    .withColumn("metadata", extrair_metadados_udf(F.col("_source_file")))
    .withColumn("seller_id_arquivo", F.col("metadata.seller_id_arquivo"))
    .withColumn("ano_arquivo", F.col("metadata.ano_arquivo"))
    .withColumn("mes_arquivo", F.col("metadata.mes_arquivo"))
    .drop("metadata")
)

qtd_arquivos = df_com_metadados.select("_source_file").distinct().count()
print(f"✅ Metadados extraídos de {qtd_arquivos} arquivo(s)")

# ========================================
# 6. PREPARAÇÃO DOS DADOS BRONZE
# ========================================

print("\n⚙️  Preparando dados para persistência...")

# IMPORTANTE: Bronze mantém dados ORIGINAIS + metadados extraídos
df_bronze = (
    df_com_metadados
    .withColumn("data_ingestao", F.current_timestamp())
)

# Validação: Verifica se conseguiu extrair metadados
registros_sem_metadados = df_bronze.filter(F.col("seller_id_arquivo").isNull()).count()
if registros_sem_metadados > 0:
    print(f"⚠️  ALERTA: {registros_sem_metadados} registros sem metadados extraídos (nome de arquivo inválido)")

# ========================================
# 7. PERSISTÊNCIA INCREMENTAL COM MERGE
# ========================================

print("\n💾 Persistindo na camada Bronze...")

# Remove duplicatas ANTES do merge (baseado em chave de negócio)
df_bronze_dedup = df_bronze.dropDuplicates(["order_id", "product_id"])

registros_novos_bronze = df_bronze_dedup.count()

if not spark.catalog.tableExists(TBL_BRONZE_VENDAS):
    print(f"🆕 Criando tabela {TBL_BRONZE_VENDAS} pela primeira vez...")
    df_bronze_dedup.write.format("delta").mode("overwrite").saveAsTable(TBL_BRONZE_VENDAS)
    registros_finais = spark.table(TBL_BRONZE_VENDAS).count()
    print(f"✅ Tabela criada com {registros_finais:,} registros")
else:
    print(f"🔄 Executando MERGE INCREMENTAL...")
    
    delta_table = DeltaTable.forName(spark, TBL_BRONZE_VENDAS)
    
    # MERGE INCREMENTAL: Insere apenas registros novos, atualiza se já existe
    # Processa APENAS os dados dos arquivos novos
    delta_table.alias("target").merge(
        df_bronze_dedup.alias("source"),
        "target.order_id = source.order_id AND target.product_id = source.product_id"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    
    registros_finais = spark.table(TBL_BRONZE_VENDAS).count()
    print(f"✅ MERGE incremental concluído. Total de registros na Bronze: {registros_finais:,}")
    print(f"   • Registros novos processados: {registros_novos_bronze:,}")

# ========================================
# 8. REGISTRO DOS ARQUIVOS PROCESSADOS
# ========================================

print("\n📝 Registrando arquivos processados na tabela de controle...")

from datetime import datetime

# Cria registro de cada arquivo processado
registros_controle = [
    (arquivo, datetime.now().isoformat(), total_registros, "processado") 
    for arquivo in arquivos_novos_list
]

df_controle_novos = spark.createDataFrame(
    registros_controle,
    ["arquivo", "data_processamento", "qtd_registros", "status"]
)

# Adiciona à tabela de controle
df_controle_novos.write.format("delta").mode("append").saveAsTable(TBL_CONTROLE_ARQUIVOS)

print(f"✅ {len(arquivos_novos_list)} arquivo(s) registrado(s) na tabela de controle")

# ========================================
# 9. VALIDAÇÃO E RESUMO
# ========================================

print("\n🔍 Validação da camada Bronze...")

df_bronze_final = spark.table(TBL_BRONZE_VENDAS)

# Verifica se todas as colunas esperadas estão presentes
colunas_esperadas = ["order_id", "product_id", "quantity", "amount", "status", "sale_date", 
                     "seller_id_arquivo", "ano_arquivo", "mes_arquivo", "data_ingestao"]

colunas_ausentes = [col for col in colunas_esperadas if col not in df_bronze_final.columns]
if colunas_ausentes:
    print(f"⚠️  ALERTA: Colunas ausentes: {', '.join(colunas_ausentes)}")
else:
    print("✅ Todas as colunas esperadas estão presentes")

print("\n" + "="*80)
print(" ✅ CAMADA BRONZE CONCLUÍDA")
print("="*80)
print(f"\n📋 Resumo:")
print(f"   • Total de registros: {registros_finais:,}")
print(f"   • Arquivos processados: {qtd_arquivos}")
print(f"   • Colunas originais preservadas: {len([c for c in df_bronze_final.columns if c not in ['seller_id_arquivo', 'ano_arquivo', 'mes_arquivo', 'data_ingestao', '_source_file']])}")
print(f"   • Metadados extraídos: seller_id_arquivo, ano_arquivo, mes_arquivo")
print(f"\n🔍 Amostra dos dados:")
display(df_bronze_final.limit(5))

In [0]:
# ================================================================================
# CAMADA SILVER - DADOS TRATADOS E VALIDADOS
# ================================================================================
# Objetivo:
# 1. Padronização de tipos e formatos
# 2. Remoção de duplicidades
# 3. Tratamento de inconsistências (produtos/vendedores sem cadastro)
# 4. Conferência de integridade entre tabelas fato/dimensões
# 5. Quarentena de registros inválidos
# ================================================================================

print("="*80)
print(" 🔷 CAMADA SILVER - TRATAMENTO E VALIDAÇÃO")
print("="*80)

# ========================================
# 1. CARREGA DADOS NOVOS DA CAMADA BRONZE
# ========================================

print("\n📂 Carregando dados NOVOS da camada Bronze...")

# Estratégia Incremental: Processa apenas registros novos (data_ingestao recente)
# Se a Silver ainda não existe, processa tudo
if spark.catalog.tableExists(TBL_SILVER_VENDAS):
    # Pega o timestamp da última execução da Silver
    ultima_exec_silver = spark.sql(f"""
        SELECT MAX(data_venda) as max_data 
        FROM {TBL_SILVER_VENDAS}
    """).collect()[0][0]
    
    if ultima_exec_silver:
        print(f"   • Última data processada na Silver: {ultima_exec_silver}")
        print(f"   • Filtrando apenas registros novos/alterados...")
        
        # Carrega apenas registros com data de ingestão maior que a última execução
        # OU registros com datas recentes (para capturar late arriving data)
        df_bronze = (
            spark.table(TBL_BRONZE_VENDAS)
            .filter(
                (F.col("data_ingestao") >= F.lit(ultima_exec_silver)) |
                (F.to_date(F.col("sale_date")) >= F.date_sub(F.lit(ultima_exec_silver), 30))  # 30 dias de look-back
            )
        )
    else:
        df_bronze = spark.table(TBL_BRONZE_VENDAS)
else:
    print("   • Silver não existe. Processando TODOS os dados da Bronze...")
    df_bronze = spark.table(TBL_BRONZE_VENDAS)

total_bronze = df_bronze.count()
print(f"✅ Bronze carregada: {total_bronze:,} registros")

# ========================================
# 2. CARREGA DIMENSÕES DE REFERÊNCIA
# ========================================

print("\n📁 Carregando dimensões de referência...")

def carregar_dimensao(caminhos, arquivo):
    """Busca e carrega um arquivo de dimensão CSV"""
    for path in caminhos:
        try:
            caminho_completo = f"{path.rstrip('/')}/{arquivo}"
            df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(caminho_completo)
            if df.limit(1).count() > 0:
                return df
        except Exception:
            continue
    raise FileNotFoundError(f"❌ {arquivo} não encontrado")

# Carrega dimensões
df_dim_produto_ref = carregar_dimensao(VOLUME_PATHS, "dim_product.csv")
df_dim_vendedor_ref = carregar_dimensao(VOLUME_PATHS, "dim_seller.csv")

print(f"✅ Produtos cadastrados: {df_dim_produto_ref.count():,}")
print(f"✅ Vendedores cadastrados: {df_dim_vendedor_ref.count():,}")

# ========================================
# 3. TRANSFORMAÇÕES E PADRONIZAÇÃO
# ========================================

print("\n⚙️  Aplicando transformações...")

# 3.1. Padroniza tipos de dados
df_transformado = (
    df_bronze
    .withColumn("id_pedido", F.col("order_id").cast("string"))
    .withColumn("id_produto", F.col("product_id").cast("string"))
    .withColumn("quantidade", F.col("quantity").cast("int"))
    .withColumn("valor_unitario", F.col("amount").cast("double"))
    .withColumn("data_venda", F.to_date(F.col("sale_date")))
    .withColumn("id_vendedor", F.col("seller_id_arquivo").cast("string"))
    # Traduz status para português
    .withColumn("status", 
        F.when(F.lower(F.col("status")) == "completed", "Concluído")
         .when(F.lower(F.col("status")) == "cancelled", "Cancelado")
         .when(F.lower(F.col("status")) == "approved", "Aprovado")
         .when(F.lower(F.col("status")) == "pending", "Pendente")
         .when(F.lower(F.col("status")) == "shipped", "Enviado")
         .when(F.lower(F.col("status")) == "delivered", "Entregue")
         .otherwise(F.col("status"))
    )
)

print("✅ Tipos padronizados e status traduzido")

# ========================================
# 4. REMOÇÃO DE DUPLICIDADES
# ========================================

print("\n🗑️  Removendo duplicidades...")

# Remove duplicatas exatas baseado em chave de negócio
df_dedup = df_transformado.dropDuplicates(["id_pedido", "id_produto"])

duplicatas_removidas = total_bronze - df_dedup.count()
if duplicatas_removidas > 0:
    print(f"⚠️  {duplicatas_removidas} duplicatas removidas")
else:
    print("✅ Nenhuma duplicata encontrada")

# ========================================
# 5. ENRIQUECIMENTO COM DIMENSÕES
# ========================================

print("\n🔗 Enriquecendo com dimensões...")

# 5.1. Enriquece com dados do produto
df_com_produto = (
    df_dedup.alias("v")
    .join(
        df_dim_produto_ref.alias("p"),
        F.col("v.id_produto") == F.col("p.product_id"),
        "left"
    )
    .withColumn("nome_produto", 
        F.coalesce(F.upper(F.col("p.product_name")), F.lit("PRODUTO SEM CADASTRO"))
    )
    .withColumn("categoria_produto",
        F.when(F.col("p.category") == "Electronics", "Eletrônicos")
         .when(F.col("p.category") == "Furniture", "Móveis")
         .when(F.col("p.category") == "Office", "Escritório")
         .otherwise(F.coalesce(F.col("p.category"), F.lit("SEM CATEGORIA")))
    )
)

# 5.2. Enriquece com dados do vendedor
df_enriquecido = (
    df_com_produto.alias("v")
    .join(
        df_dim_vendedor_ref.alias("s"),
        F.col("v.id_vendedor") == F.col("s.seller_id").cast("string"),
        "left"
    )
    .withColumn("nome_vendedor",
        F.coalesce(F.col("s.seller_name"), F.lit("VENDEDOR SEM CADASTRO"))
    )
    .withColumn("estado_vendedor",
        F.coalesce(F.col("s.state"), F.lit("N/A"))
    )
)

print("✅ Enriquecimento concluído")

# ========================================
# 6. SELEÇÃO DE COLUNAS FINAIS
# ========================================

df_silver_prep = df_enriquecido.select(
    "id_pedido",
    "id_produto",
    "nome_produto",
    "categoria_produto",
    "id_vendedor",
    "nome_vendedor",
    "estado_vendedor",
    "quantidade",
    "valor_unitario",
    F.round(F.col("quantidade") * F.col("valor_unitario"), 2).alias("valor_total"),
    "status",
    "data_venda",
    F.year("data_venda").alias("ano"),
    F.month("data_venda").alias("mes"),
    F.quarter("data_venda").alias("trimestre")
)

# ========================================
# 7. VALIDAÇÃO E QUARENTENA
# ========================================

print("\n🔍 Validando dados...")

# Critérios de validação:
# - Quantidade deve ser numérica e positiva
# - Valor unitário deve ser numérico e positivo
# - Data de venda deve estar presente

df_validos = df_silver_prep.filter(
    (F.col("quantidade").isNotNull()) &
    (F.col("valor_unitario").isNotNull()) &
    (F.col("data_venda").isNotNull()) &
    (F.col("quantidade") > 0) &
    (F.col("valor_unitario") > 0)
)

# Identifica registros inválidos (para quarentena)
df_invalidos = df_silver_prep.join(
    df_validos.select("id_pedido", "id_produto"),
    ["id_pedido", "id_produto"],
    "left_anti"
).withColumn("motivo_quarentena",
    F.when(F.col("quantidade").isNull() | (F.col("quantidade") <= 0), "Quantidade inválida")
     .when(F.col("valor_unitario").isNull() | (F.col("valor_unitario") <= 0), "Valor inválido")
     .when(F.col("data_venda").isNull(), "Data ausente")
     .otherwise("Outros")
).withColumn("data_quarentena", F.current_timestamp())

qtd_validos = df_validos.count()
qtd_invalidos = df_invalidos.count()

print(f"✅ Registros válidos: {qtd_validos:,}")
print(f"⚠️  Registros inválidos (quarentena): {qtd_invalidos:,}")

# ========================================
# 8. PERSISTÊNCIA INCREMENTAL
# ========================================

print("\n💾 Persistindo camada Silver com MERGE incremental...")

# MERGE INCREMENTAL: Atualiza registros existentes, insere novos
if not spark.catalog.tableExists(TBL_SILVER_VENDAS):
    print(f"   • Criando tabela Silver pela primeira vez...")
    df_validos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_SILVER_VENDAS)
else:
    print(f"   • Executando MERGE incremental...")
    delta_table_silver = DeltaTable.forName(spark, TBL_SILVER_VENDAS)
    
    # MERGE: Atualiza se já existe, insere se novo
    delta_table_silver.alias("target").merge(
        df_validos.alias("source"),
        "target.id_pedido = source.id_pedido AND target.id_produto = source.id_produto"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print(f"   • MERGE concluído")

# Salva registros inválidos na quarentena (se houver)
if qtd_invalidos > 0:
    if not spark.catalog.tableExists(TBL_QUARENTENA):
        df_invalidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_QUARENTENA)
    else:
        # MERGE na quarentena também
        delta_table_quarentena = DeltaTable.forName(spark, TBL_QUARENTENA)
        delta_table_quarentena.alias("target").merge(
            df_invalidos.alias("source"),
            "target.id_pedido = source.id_pedido AND target.id_produto = source.id_produto"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
    print(f"⚠️  Quarentena atualizada: {TBL_QUARENTENA}")

# ========================================
# 9. RELATÓRIO DE QUALIDADE
# ========================================

print("\n" + "="*80)
print(" ✅ CAMADA SILVER CONCLUÍDA")
print("="*80)

df_silver_final = spark.table(TBL_SILVER_VENDAS)

print(f"\n📋 Relatório de Qualidade:")
print(f"   • Registros Bronze: {total_bronze:,}")
print(f"   • Duplicatas removidas: {duplicatas_removidas:,}")
print(f"   • Registros válidos (Silver): {qtd_validos:,}")
print(f"   • Registros em quarentena: {qtd_invalidos:,}")
print(f"   • Taxa de qualidade: {(qtd_validos/total_bronze*100):.2f}%")

# Estatísticas de enriquecimento
produtos_sem_cadastro = df_silver_final.filter(F.col("nome_produto") == "PRODUTO SEM CADASTRO").count()
vendedores_sem_cadastro = df_silver_final.filter(F.col("nome_vendedor") == "VENDEDOR SEM CADASTRO").count()

print(f"\n🗒️  Análise de Cobertura:")
print(f"   • Produtos sem cadastro: {produtos_sem_cadastro:,} ({(produtos_sem_cadastro/qtd_validos*100):.2f}%)")
print(f"   • Vendedores sem cadastro: {vendedores_sem_cadastro:,} ({(vendedores_sem_cadastro/qtd_validos*100):.2f}%)")

print(f"\n🔍 Amostra Silver:")
display(df_silver_final.limit(5))

In [0]:
# ================================================================================
# CAMADA GOLD - MODELAGEM STAR SCHEMA
# ================================================================================
# Objetivo:
# Criar modelo dimensional para consumo analítico (Power BI, Excel, etc.)
#
# Estrutura:
# - Tabela Fato: fato_vendas (fatos das transações)
# - Dimensões: dim_produto, dim_vendedor, dim_tempo
#
# Justificativa do Star Schema:
# 1. Facilidade de entendimento: Estrutura intuitiva (fato + dimensões)
# 2. Performance: Joins simples e diretos, ideais para queries analíticas
# 3. Manutenção: Separação clara entre fatos e dimensões
# 4. Escalabilidade: Novas dimensões podem ser adicionadas sem alterar o fato
# ================================================================================

print("="*80)
print(" 🌟 CAMADA GOLD - STAR SCHEMA")
print("="*80)

# ========================================
# 1. CARREGA CAMADA SILVER
# ========================================

print("\n📂 Carregando Silver...")
df_silver = spark.table(TBL_SILVER_VENDAS)
total_silver = df_silver.count()
print(f"✅ {total_silver:,} registros carregados")

# ========================================
# 2. DIMENSÃO PRODUTO
# ========================================

print("\n📦 Criando Dimensão Produto...")

df_dim_produto = (
    df_silver
    .select("id_produto", "nome_produto", "categoria_produto")
    .dropDuplicates(["id_produto"])
    .withColumn("sk_produto", F.col("id_produto"))  # Surrogate key
    .select(
        "sk_produto",
        "id_produto",
        "nome_produto",
        "categoria_produto"
    )
)

# MERGE INCREMENTAL: Atualiza dimensões existentes, insere novas
if not spark.catalog.tableExists(TBL_DIM_PRODUTO):
    df_dim_produto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_DIM_PRODUTO)
else:
    delta_dim_produto = DeltaTable.forName(spark, TBL_DIM_PRODUTO)
    delta_dim_produto.alias("target").merge(
        df_dim_produto.alias("source"),
        "target.id_produto = source.id_produto"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()

qtd_produtos = spark.table(TBL_DIM_PRODUTO).count()
print(f"✅ {TBL_DIM_PRODUTO} atualizada: {qtd_produtos:,} produtos")

# ========================================
# 3. DIMENSÃO VENDEDOR (COMPLETA COM STATUS)
# ========================================

print("\n👥 Criando Dimensão Vendedor...")

# Carrega TODOS os vendedores do cadastro (dim_seller.csv)
print("   • Carregando cadastro completo de vendedores...")
df_vendedores_cadastro = carregar_dimensao(VOLUME_PATHS, "dim_seller.csv")

# Identifica vendedores que têm vendas (aparecem na Silver)
df_vendedores_com_vendas = (
    df_silver
    .select("id_vendedor")
    .distinct()
    .withColumn("tem_vendas", F.lit(True))
)

# Cria dimensão completa: TODOS os vendedores do cadastro + status
df_dim_vendedor = (
    df_vendedores_cadastro
    .withColumn("id_vendedor", F.col("seller_id").cast("string"))
    .join(
        df_vendedores_com_vendas,
        "id_vendedor",
        "left"
    )
    .withColumn("status_vendedor",
        F.when(F.col("tem_vendas") == True, "Recorrente")
         .otherwise("Novo")
    )
    .withColumn("sk_vendedor", F.col("id_vendedor"))
    .select(
        "sk_vendedor",
        "id_vendedor",
        F.col("seller_name").alias("nome_vendedor"),
        F.col("state").alias("estado_vendedor"),
        "status_vendedor"
    )
)

qtd_novos = df_dim_vendedor.filter(F.col("status_vendedor") == "Novo").count()
qtd_recorrentes = df_dim_vendedor.filter(F.col("status_vendedor") == "Recorrente").count()
print(f"   • Vendedores Novos (sem vendas): {qtd_novos}")
print(f"   • Vendedores Recorrentes (com vendas): {qtd_recorrentes}")

# Recria a tabela com a nova estrutura (inclui coluna status_vendedor)
df_dim_vendedor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_DIM_VENDEDOR)

qtd_vendedores = spark.table(TBL_DIM_VENDEDOR).count()
qtd_novos = spark.table(TBL_DIM_VENDEDOR).filter(F.col("status_vendedor") == "Novo").count()
qtd_recorrentes = spark.table(TBL_DIM_VENDEDOR).filter(F.col("status_vendedor") == "Recorrente").count()

print(f"✅ {TBL_DIM_VENDEDOR} atualizada: {qtd_vendedores:,} vendedores")
print(f"   • Novos (sem vendas): {qtd_novos}")
print(f"   • Recorrentes (com vendas): {qtd_recorrentes}")

# ========================================
# 4. DIMENSÃO TEMPO
# ========================================

print("\n📅 Criando Dimensão Tempo...")

df_dim_tempo = (
    df_silver
    .select("data_venda")
    .dropDuplicates(["data_venda"])
    .withColumn("sk_tempo", F.date_format(F.col("data_venda"), "yyyyMMdd").cast("int"))  # Ex: 20240115
    .withColumn("ano", F.year("data_venda"))
    .withColumn("mes", F.month("data_venda"))
    .withColumn("trimestre", F.quarter("data_venda"))
    .withColumn("dia", F.dayofmonth("data_venda"))
    .withColumn("dia_semana", F.dayofweek("data_venda"))
    .withColumn("nome_mes",
        F.when(F.col("mes") == 1, "Janeiro")
         .when(F.col("mes") == 2, "Fevereiro")
         .when(F.col("mes") == 3, "Março")
         .when(F.col("mes") == 4, "Abril")
         .when(F.col("mes") == 5, "Maio")
         .when(F.col("mes") == 6, "Junho")
         .when(F.col("mes") == 7, "Julho")
         .when(F.col("mes") == 8, "Agosto")
         .when(F.col("mes") == 9, "Setembro")
         .when(F.col("mes") == 10, "Outubro")
         .when(F.col("mes") == 11, "Novembro")
         .when(F.col("mes") == 12, "Dezembro")
    )
    .withColumn("nome_dia_semana",
        F.when(F.col("dia_semana") == 1, "Domingo")
         .when(F.col("dia_semana") == 2, "Segunda")
         .when(F.col("dia_semana") == 3, "Terça")
         .when(F.col("dia_semana") == 4, "Quarta")
         .when(F.col("dia_semana") == 5, "Quinta")
         .when(F.col("dia_semana") == 6, "Sexta")
         .when(F.col("dia_semana") == 7, "Sábado")
    )
    .select(
        "sk_tempo",
        "data_venda",
        "ano",
        "mes",
        "nome_mes",
        "trimestre",
        "dia",
        "dia_semana",
        "nome_dia_semana"
    )
)

# MERGE INCREMENTAL
if not spark.catalog.tableExists(TBL_DIM_TEMPO):
    df_dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_DIM_TEMPO)
else:
    delta_dim_tempo = DeltaTable.forName(spark, TBL_DIM_TEMPO)
    delta_dim_tempo.alias("target").merge(
        df_dim_tempo.alias("source"),
        "target.sk_tempo = source.sk_tempo"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()

qtd_datas = spark.table(TBL_DIM_TEMPO).count()
print(f"✅ {TBL_DIM_TEMPO} atualizada: {qtd_datas:,} datas distintas")

# ========================================
# 5. TABELA FATO VENDAS
# ========================================

print("\n📊 Criando Tabela Fato de Vendas...")

df_fato_vendas = (
    df_silver
    .withColumn("sk_fato", F.monotonically_increasing_id())  # Surrogate key do fato
    .withColumn("sk_tempo", F.date_format(F.col("data_venda"), "yyyyMMdd").cast("int"))
    .select(
        "sk_fato",
        "id_pedido",
        F.col("id_produto").alias("fk_produto"),
        F.col("id_vendedor").alias("fk_vendedor"),
        "sk_tempo",
        "quantidade",
        "valor_unitario",
        "valor_total",
        "status",
        "data_venda"
    )
)

# MERGE INCREMENTAL no Fato
if not spark.catalog.tableExists(TBL_FATO_VENDAS):
    df_fato_vendas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_FATO_VENDAS)
else:
    delta_fato = DeltaTable.forName(spark, TBL_FATO_VENDAS)
    delta_fato.alias("target").merge(
        df_fato_vendas.alias("source"),
        "target.id_pedido = source.id_pedido AND target.fk_produto = source.fk_produto"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()

qtd_fatos = spark.table(TBL_FATO_VENDAS).count()
print(f"✅ {TBL_FATO_VENDAS} atualizada: {qtd_fatos:,} transações")

# ========================================
# 6. VALIDAÇÃO DE INTEGRIDADE REFERENCIAL
# ========================================

print("\n🔍 Validando integridade referencial...")

df_fato = spark.table(TBL_FATO_VENDAS)
df_dim_prod = spark.table(TBL_DIM_PRODUTO)
df_dim_vend = spark.table(TBL_DIM_VENDEDOR)
df_dim_time = spark.table(TBL_DIM_TEMPO)

# Valida produtos órfãos (fatos sem dimensão correspondente)
produtos_orfaos = df_fato.join(
    df_dim_prod,
    df_fato.fk_produto == df_dim_prod.id_produto,
    "left_anti"
).count()

vendedores_orfaos = df_fato.join(
    df_dim_vend,
    df_fato.fk_vendedor == df_dim_vend.id_vendedor,
    "left_anti"
).count()

tempo_orfaos = df_fato.join(
    df_dim_time,
    df_fato.sk_tempo == df_dim_time.sk_tempo,
    "left_anti"
).count()

if produtos_orfaos + vendedores_orfaos + tempo_orfaos == 0:
    print("✅ Integridade referencial OK! Todos os fatos possuem dimensões correspondentes")
else:
    print(f"⚠️  ALERTA: Problemas de integridade:")
    if produtos_orfaos > 0:
        print(f"   - {produtos_orfaos} vendas com produtos sem dimensão")
    if vendedores_orfaos > 0:
        print(f"   - {vendedores_orfaos} vendas com vendedores sem dimensão")
    if tempo_orfaos > 0:
        print(f"   - {tempo_orfaos} vendas com datas sem dimensão")

# ========================================
# 7. RESUMO DO STAR SCHEMA
# ========================================

print("\n" + "="*80)
print(" ✅ CAMADA GOLD - STAR SCHEMA CONCLUÍDO")
print("="*80)

print(f"\n🌟 Estrutura do Star Schema:")
print(f"\n   📊 TABELA FATO:")
print(f"      • {TBL_FATO_VENDAS}: {qtd_fatos:,} transações")
print(f"\n   📁 DIMENSÕES:")
print(f"      • {TBL_DIM_PRODUTO}: {qtd_produtos:,} produtos")
print(f"      • {TBL_DIM_VENDEDOR}: {qtd_vendedores:,} vendedores")
print(f"      • {TBL_DIM_TEMPO}: {qtd_datas:,} datas")

print(f"\n📊 Estatísticas de Vendas:")
df_stats = spark.table(TBL_FATO_VENDAS)

receita_total = df_stats.filter(F.col("status").isin(["Concluído", "Aprovado"])).agg(F.sum("valor_total")).collect()[0][0]
qtd_vendas_validas = df_stats.filter(F.col("status").isin(["Concluído", "Aprovado"])).count()
ticket_medio = df_stats.filter(F.col("status").isin(["Concluído", "Aprovado"])).agg(F.avg("valor_total")).collect()[0][0]

print(f"   • Receita Total (vendas concluídas/aprovadas): R$ {receita_total:,.2f}")
print(f"   • Quantidade de vendas válidas: {qtd_vendas_validas:,}")
print(f"   • Ticket médio: R$ {ticket_medio:,.2f}")

print(f"\n🔍 Amostra do Fato de Vendas (com joins):")

df_amostra = (
    spark.table(TBL_FATO_VENDAS).alias("f")
    .join(spark.table(TBL_DIM_PRODUTO).alias("p"), F.col("f.fk_produto") == F.col("p.id_produto"))
    .join(spark.table(TBL_DIM_VENDEDOR).alias("v"), F.col("f.fk_vendedor") == F.col("v.id_vendedor"))
    .join(spark.table(TBL_DIM_TEMPO).alias("t"), F.col("f.sk_tempo") == F.col("t.sk_tempo"))
    .select(
        "f.id_pedido",
        "p.nome_produto",
        "p.categoria_produto",
        "v.nome_vendedor",
        "v.estado_vendedor",
        "f.quantidade",
        "f.valor_total",
        "f.status",
        "t.data_venda",
        "t.nome_mes"
    )
    .limit(5)
)

display(df_amostra)

In [0]:
# ================================================================================
# MÉTRICAS AGREGADAS E VALIDAÇÃO DE QUALIDADE
# ================================================================================
# Objetivo:
# 1. Criar tabela de métricas pré-calculadas para consumo rápido em BI
# 2. Documentar estratégias de validação de qualidade
# 3. Demonstrar idempotência e controle de reprocessamento
# ================================================================================

print("="*80)
print(" 📊 MÉTRICAS AGREGADAS E VALIDAÇÃO FINAL")
print("="*80)

# ========================================
# 1. TABELA DE MÉTRICAS PRÉ-CALCULADAS
# ========================================

print("\n📊 Criando tabela de métricas pré-calculadas...")

df_metricas = (
    spark.table(TBL_FATO_VENDAS).alias("f")
    .join(spark.table(TBL_DIM_PRODUTO).alias("p"), F.col("f.fk_produto") == F.col("p.id_produto"))
    .join(spark.table(TBL_DIM_VENDEDOR).alias("v"), F.col("f.fk_vendedor") == F.col("v.id_vendedor"))
    .join(spark.table(TBL_DIM_TEMPO).alias("t"), F.col("f.sk_tempo") == F.col("t.sk_tempo"))
    .filter(F.col("f.status").isin(["Concluído", "Aprovado"]))  # Apenas vendas válidas
    .groupBy(
        F.col("t.ano"),
        F.col("t.mes"),
        F.col("t.nome_mes"),
        F.col("p.categoria_produto"),
        F.col("v.estado_vendedor")
    )
    .agg(
        F.count("*").alias("qtd_vendas"),
        F.sum("f.quantidade").alias("qtd_itens_vendidos"),
        F.round(F.sum("f.valor_total"), 2).alias("receita_total"),
        F.round(F.avg("f.valor_total"), 2).alias("ticket_medio"),
        F.round(F.min("f.valor_total"), 2).alias("ticket_minimo"),
        F.round(F.max("f.valor_total"), 2).alias("ticket_maximo"),
        F.countDistinct("f.fk_produto").alias("qtd_produtos_distintos"),
        F.countDistinct("f.fk_vendedor").alias("qtd_vendedores_ativos")
    )
    .orderBy("ano", "mes", "categoria_produto", "estado_vendedor")
)

df_metricas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_METRICAS_VENDAS)
qtd_metricas = df_metricas.count()
print(f"✅ {TBL_METRICAS_VENDAS} criada: {qtd_metricas:,} combinações de métricas")

# ========================================
# 2. VALIDAÇÃO DE QUALIDADE - CHECKS FINAIS
# ========================================

print("\n" + "="*80)
print(" 🔍 VALIDAÇÃO DE QUALIDADE DOS DADOS")
print("="*80)

print("\n📋 Checks de Qualidade:")

# Check 1: Total de registros em cada camada
qtd_bronze = spark.table(TBL_BRONZE_VENDAS).count()
qtd_silver = spark.table(TBL_SILVER_VENDAS).count()
qtd_fato = spark.table(TBL_FATO_VENDAS).count()

print(f"\n1️⃣  Contagem de Registros:")
print(f"   • Bronze: {qtd_bronze:,} registros")
print(f"   • Silver: {qtd_silver:,} registros")
print(f"   • Gold (Fato): {qtd_fato:,} registros")
print(f"   ✅ Consistência: Silver = Fato? {qtd_silver == qtd_fato}")

# Check 2: Valores nulos em colunas críticas
print(f"\n2️⃣  Valores Nulos em Colunas Críticas (Silver):")
df_silver_check = spark.table(TBL_SILVER_VENDAS)

colunas_criticas = ["id_pedido", "id_produto", "quantidade", "valor_unitario", "data_venda"]
for col in colunas_criticas:
    nulos = df_silver_check.filter(F.col(col).isNull()).count()
    if nulos > 0:
        print(f"   ⚠️  {col}: {nulos:,} nulos encontrados")
    else:
        print(f"   ✅ {col}: Sem nulos")

# Check 3: Valores negativos ou zero
print(f"\n3️⃣  Valores Inválidos (negativos ou zero):")
qtd_negativas = df_silver_check.filter((F.col("quantidade") <= 0) | (F.col("valor_unitario") <= 0)).count()
print(f"   {'\u2705 Nenhum valor inv\u00e1lido' if qtd_negativas == 0 else f'\u26a0\ufe0f  {qtd_negativas:,} registros com valores inv\u00e1lidos'}")

# Check 4: Duplicatas
print(f"\n4️⃣  Duplicatas:")
qtd_total = df_silver_check.count()
qtd_distintos = df_silver_check.dropDuplicates(["id_pedido", "id_produto"]).count()
duplicatas = qtd_total - qtd_distintos
print(f"   {'\u2705 Nenhuma duplicata' if duplicatas == 0 else f'\u26a0\ufe0f  {duplicatas:,} duplicatas encontradas'}")

# Check 5: Distribuição de Status
print(f"\n5️⃣  Distribuição de Status:")
df_status = spark.table(TBL_FATO_VENDAS).groupBy("status").count().orderBy(F.desc("count"))
for row in df_status.collect():
    print(f"   • {row['status']}: {row['count']:,} ({row['count']/qtd_fato*100:.2f}%)")

# Check 6: Cobertura de Dimensões
print(f"\n6️⃣  Cobertura de Dimensões:")
produtos_sem_cadastro_pct = spark.table(TBL_SILVER_VENDAS).filter(F.col("nome_produto") == "PRODUTO SEM CADASTRO").count() / qtd_silver * 100
vendedores_sem_cadastro_pct = spark.table(TBL_SILVER_VENDAS).filter(F.col("nome_vendedor") == "VENDEDOR SEM CADASTRO").count() / qtd_silver * 100

print(f"   • Produtos sem cadastro: {produtos_sem_cadastro_pct:.2f}%")
print(f"   • Vendedores sem cadastro: {vendedores_sem_cadastro_pct:.2f}%")
if produtos_sem_cadastro_pct < 5 and vendedores_sem_cadastro_pct < 5:
    print(f"   ✅ Cobertura adequada (<5% sem cadastro)")
else:
    print(f"   ⚠️  Cobertura insuficiente (>5% sem cadastro)")

# ========================================
# 3. DOCUMENTAÇÃO DE ESTRATÉGIAS
# ========================================

print("\n" + "="*80)
print(" 📖 ESTRATÉGIAS DE QUALIDADE E CONFIABILIDADE")
print("="*80)

print("""
🔍 COMO VALIDAR A QUALIDADE DOS DADOS APÓS O TRATAMENTO:

1. Contagem de Registros:
   • Verificar se o número de registros bate entre camadas
   • Silver = Gold (Fato) indica consistência
   • Diferenças esperadas: Bronze > Silver (devido a remoção de inválidos)

2. Validação de Nulos:
   • Colunas críticas (id_pedido, quantidade, valor, data) não devem ter nulos
   • Registros com nulos vão para quarentena

3. Validação de Valores:
   • Quantidade e valores devem ser positivos
   • Datas devem estar em range válido

4. Remoção de Duplicatas:
   • Chave de negócio: (id_pedido, id_produto)
   • Duplicatas removidas na Silver

5. Totais Financeiros:
   • Soma(Bronze.valor_total) ≈ Soma(Silver.valor_total) - quarentena
   • Divergências acima de 1% devem ser investigadas

🔍 COMO IDENTIFICAR E SINALIZAR ARQUIVOS COM PROBLEMA:

1. Schema Inesperado:
   • Validar colunas esperadas vs colunas recebidas
   • Logar arquivos com colunas faltando ou extras
   • Usar option("mode", "PERMISSIVE") para capturar registros malformados

2. Arquivo Vazio:
   • Verificar count() após leitura
   • Alertar se count == 0

3. Valores Fora do Padrão:
   • Usar regras de negócio (ex: preço > 0, quantidade > 0)
   • Enviar para quarentena registros inválidos
   • Gerar alerta se % quarentena > threshold (ex: 10%)

4. Metadados do Arquivo:
   • Nome do arquivo deve seguir padrão: SELLER_ID_YYYY_MM_sales.csv
   • Arquivos fora do padrão devem ser rejeitados ou alertados

🔍 COMO EVITAR CONTAR A MESMA VENDA DUAS VEZES:

1. IDEMPOTÊNCIA COM MERGE:
   • Usar Delta MERGE ao invés de INSERT simples
   • Chave de comparação: (order_id, product_id)
   • whenMatchedUpdateAll: Atualiza se existir
   • whenNotMatchedInsertAll: Insere se novo
   • Resultado: Rodar o Job 2x nos mesmos arquivos NÃO duplica dados

2. DEDUPLICATION:
   • Usar dropDuplicates(["id_pedido", "id_produto"]) antes de persistência
   • Remove duplicatas dentro do mesmo batch

3. CONTROLE DE PROCESSAMENTO:
   • Manter tabela de controle com arquivos já processados
   • Registrar: nome_arquivo, data_processamento, qtd_registros
   • Filtrar arquivos já processados antes da ingestão
   
4. WATERMARK DE TIMESTAMP:
   • Adicionar coluna "data_ingestao" em todas as camadas
   • Permite rastrear quando cada registro foi processado
   • Útil para auditoria e troubleshooting

📊 ESTRATÉGIA DE REEXECUÇÃO EM CASO DE FALHA:

1. Dependência entre Etapas:
   • Bronze → Silver → Gold
   • Se Bronze falha: reprocessar apenas Bronze
   • Se Silver falha: reprocessar Silver + Gold
   • Se Gold falha: reprocessar apenas Gold

2. Recuperação:
   • Delta Lake permite rollback via Time Travel
   • Ex: RESTORE TABLE bronze_vendas TO VERSION AS OF 10
   • Ex: SELECT * FROM bronze_vendas@v10

3. Retry Automático:
   • Configurar no Job: max_retries = 2
   • Retry com backoff exponencial
   • Alertar equipe após todas as tentativas falharem

4. Monitoração:
   • Alertas em tempo real para falhas
   • Métricas: tempo de execução, qtd de registros, taxa de sucesso
   • Dashboard com histórico de execuções
""")

# ========================================
# 4. RESUMO FINAL
# ========================================

print("\n" + "="*80)
print(" ✅ PIPELINE CONCLUÍDO COM SUCESSO")
print("="*80)

print(f"\n🎉 Resumo Geral:")
print(f"\n   🔷 BRONZE:")
print(f"      • Tabela: {TBL_BRONZE_VENDAS}")
print(f"      • Registros: {qtd_bronze:,}")
print(f"      • Idempotência: Garantida via MERGE")

print(f"\n   🔷 SILVER:")
print(f"      • Tabela: {TBL_SILVER_VENDAS}")
print(f"      • Registros: {qtd_silver:,}")
print(f"      • Qualidade: {(qtd_silver/qtd_bronze*100):.2f}% de aprovação")
if spark.catalog.tableExists(TBL_QUARENTENA):
    qtd_quarentena = spark.table(TBL_QUARENTENA).count()
    print(f"      • Quarentena: {qtd_quarentena:,} registros")

print(f"\n   🌟 GOLD:")
print(f"      • Fato: {TBL_FATO_VENDAS} ({qtd_fato:,} transações)")
print(f"      • Dim Produto: {spark.table(TBL_DIM_PRODUTO).count():,} produtos")
print(f"      • Dim Vendedor: {spark.table(TBL_DIM_VENDEDOR).count():,} vendedores")
print(f"      • Dim Tempo: {spark.table(TBL_DIM_TEMPO).count():,} datas")
print(f"      • Métricas: {qtd_metricas:,} combinações pré-calculadas")

print(f"\n🎯 Próximos Passos:")
print(f"   1. Conectar Power BI / Excel às tabelas Gold")
print(f"   2. Criar Job de Orquestração (Bronze → Silver → Gold)")
print(f"   3. Configurar alertas de falha e monitoração")
print(f"   4. Implementar incrementalização (processar apenas novos arquivos)")

print(f"\n📊 Amostra de Métricas:")
display(spark.table(TBL_METRICAS_VENDAS).limit(10))